In [ ]:
# Run from the repository root so data/, scripts/, dashboard/, outputs/ paths resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Mechanistic video corroboration: analysis

Reproduces the analysis of the blinded antecedent video coding (`CODING_SHEET_antecedent_COMPLETE.csv`)
merged with the unblind key (`blind_clips/_UNBLIND_KEY_antecedent.csv`).

**Design.** Set A = Route-2 squirrel events *with* a contact; B = squirrel *no* contact; C = Route-2
Name; D = Route-3 Sign+Animal (no crashes).

**HUD caveat (important).** The Name target is a heads-up-display object with no world location, so
Set C *cannot* exhibit `approach_to_board` by construction. C is therefore **not** a valid comparator
for the approach / dual-zoom codes, only for eyes-off-road (telemetry road-gaze). The verbal/visual
*target-type specificity* is tested with telemetry `During_Road_Gaze_Pct`, not video approach.

**Packages & method.** pandas / numpy for the blinded-coding merge and event rates; scipy `fisher_exact` for the marker–collision associations (an exact test, appropriate for the small 2×2 counts) and `mannwhitneyu` for the telemetry road-gaze contrast (its AUC is the reported effect size).

In [1]:
import numpy as np, pandas as pd
from scipy.stats import fisher_exact, mannwhitneyu, spearmanr

CODED = "data/CODING_SHEET_antecedent_COMPLETE.csv"
KEY   = "data/blind_clips/_UNBLIND_KEY_antecedent.csv"
TELE  = "data/master_routes_1_to_6_latency.csv"

coded = pd.read_csv(CODED, dtype=str); coded.columns = [c.strip() for c in coded.columns]
key   = pd.read_csv(KEY)
df = coded.merge(key, on="blind_id", validate="1:1")
print("merged", len(df), "clips | sets:", df.set.value_counts().to_dict())
assert (key.cue_s - key.start_s).round(2).eq(2.0).all(), "cue offset is not uniformly 2.0s"
print("cue offset uniform at 2.0s -> latency = first_gaze_s - 2.0")

merged 148 clips | sets: {'D': 74, 'C': 37, 'B': 23, 'A': 14}
cue offset uniform at 2.0s -> latency = first_gaze_s - 2.0


## 1. Cleaning rules (pre-agreed)

* `first_gaze_s`: comma-decimal fixed; `GLITCH` rows excluded; `NA`/blank → missing.
* `latency_s` = entered latency if present, else `first_gaze_s − 2.0` (pre-cue reads < 2.0 clamped to 0).
* Manual overrides: precue-on-target rows with no time (`27EA51`,`A8B537`,`B41B2C`) → 0; `156AFE` → NA.
* `no_acquisition` = never fixated the target (latency NA & not a working-memory Name answer), kept as
  a meaningful SA-failure category, not dropped.
* `approach_to_board` was coded **binary 0/1** (the 0–3 gradient lives in the notes), so we use
  `approach = (approach_to_board ≥ 1)`.

In [2]:
def num(x):
    if x is None: return np.nan
    s = str(x).strip().replace(',', '.')
    if s in ('','NA','nan','NaN'): return np.nan
    if s.upper()=='GLITCH': return 'GLITCH'
    try: return float(s)
    except: return np.nan

df['fg_raw']      = df['first_gaze_s'].map(num)
df['lat_entered'] = df['gaze_to_stimulus_latency_s'].map(num)
df['GLITCH']      = df['fg_raw'].astype(str).eq('GLITCH')

def latency(r):
    if r['GLITCH']: return np.nan
    le = r['lat_entered']
    if isinstance(le,(int,float)) and not pd.isna(le): return float(le)
    fg = r['fg_raw']
    if isinstance(fg,(int,float)) and not pd.isna(fg): return max(0.0, fg-2.0)
    return np.nan
df['latency_s'] = df.apply(latency, axis=1)

# manual overrides
df.loc[df.blind_id.isin(['CLIP_27EA51','CLIP_A8B537','CLIP_B41B2C']), 'latency_s'] = 0.0
df.loc[df.blind_id=='CLIP_156AFE', 'latency_s'] = np.nan

def ordn(x):
    s=str(x).strip()
    if s in ('','NA','nan'): return np.nan
    try: return float(s)
    except: return np.nan
for c in ['precue_fixation_0_1','apparent_lean_in_0_1','approach_to_board_0_3','line_departure_0_2']:
    df[c+'_n'] = df[c].map(ordn)

v = df[~df.GLITCH].copy()
v['approach'] = (v.approach_to_board_0_3_n>=1).astype(float)
v['lean']     = v.apparent_lean_in_0_1_n
v['linedep']  = (v.line_departure_0_2_n>=1).astype(float)
v['dualzoom'] = ((v.lean==1)&(v.approach==1)).astype(float)
v['visualID'] = v.set.isin(['A','B','D'])
v['no_acquisition'] = v.latency_s.isna() & ~v.notes.fillna('').str.contains('memory', case=False)

print('excluded GLITCH:', int(df.GLITCH.sum()), '| analysed:', len(v))
print('no-acquisition by set:', {s:int(v[v.set==s].no_acquisition.sum()) for s in 'ABCD'})

excluded GLITCH: 6 | analysed: 142
no-acquisition by set: {'A': 2, 'B': 2, 'C': 0, 'D': 3}


## 2. Descriptives by set

In [3]:
def rate(s): return f"{100*np.nanmean(s):.0f}%"
def setrow(g):
    return pd.Series({'n':len(g),
        'approach':rate(g.approach), 'lean':rate(g.lean),
        'line_dep':rate(g.linedep), 'dual_zoom':rate(g.dualzoom)})
v.groupby('set').apply(setrow, include_groups=False)

,n,approach,lean,line_dep,dual_zoom
set,,,,,
A,13,69%,46%,92%,46%
B,23,26%,30%,52%,17%
C,36,0%,6%,11%,0%
D,70,36%,30%,43%,20%


## 3. Core test: A vs B (crash vs no-crash, identical squirrel)

Same world-object stimulus, so **no HUD confound**. Tests whether contacts co-occur with more of the
approach / detachment behaviour.

In [4]:
def fisher_tbl(g1,g2,col):
    a1=int(np.nansum(g1[col]==1)); n1=int(g1[col].notna().sum())
    a2=int(np.nansum(g2[col]==1)); n2=int(g2[col].notna().sum())
    _,p=fisher_exact([[a1,n1-a1],[a2,n2-a2]])
    return {'A':f"{a1}/{n1} ({100*a1/n1:.0f}%)", 'B':f"{a2}/{n2} ({100*a2/n2:.0f}%)", 'p':round(p,4)}
A,B = v[v.set=='A'], v[v.set=='B']
pd.DataFrame({c:fisher_tbl(A,B,c) for c in ['approach','lean','linedep','dualzoom']}).T

,A,B,p
approach,9/13 (69%),6/23 (26%),0.0168
lean,6/13 (46%),7/23 (30%),0.4741
linedep,12/13 (92%),12/23 (52%),0.0253
dualzoom,6/13 (46%),4/23 (17%),0.1194


## 4. D: mechanism without contact (Route 3, descriptive)

Route-3 visual-ID events show the behaviour with **no** collisions, so it is not a Route-2 artefact.

In [5]:
D = v[v.set=='D']
pd.Series({c:f"{int(np.nansum(D[c]==1))}/{int(D[c].notna().sum())} ({100*np.nanmean(D[c]):.0f}%)"
           for c in ['approach','lean','linedep','dualzoom']}, name='Route-3 rate').to_frame()

,Route-3 rate
approach,25/70 (36%)
lean,21/69 (30%)
linedep,30/70 (43%)
dualzoom,14/70 (20%)


## 5. Target-type specificity = telemetry road-gaze (valid for a HUD)

Because Set C cannot show video approach (HUD), the verbal/visual discriminant uses the independent
eye-tracking metric `During_Road_Gaze_Pct`.

In [6]:
t = pd.read_csv(TELE)[['Participant_ID','Route','Event','During_Diffuse_Gaze_Pct','During_Mean_Speed_mph','Target_Latency']]
m = v.merge(t, left_on=['pid','route','event'], right_on=['Participant_ID','Route','Event'], how='left')
print('telemetry matched:', m.During_Diffuse_Gaze_Pct.notna().sum(), '/', len(m))
world = m[m.set.isin(['A','B','D'])].During_Diffuse_Gaze_Pct.dropna()
name  = m[m.set=='C'].During_Diffuse_Gaze_Pct.dropna()
u,p = mannwhitneyu(world, name)
print(f"diffuse off-object gaze: world visual-ID median={world.median():.2f} (n={len(world)})  "
      f"vs Name/HUD median={name.median():.2f} (n={len(name)})   p={p:.4g}")

telemetry matched: 142 / 142
road-gaze: world visual-ID median=0.47 (n=106)  vs Name/HUD median=0.90 (n=36)   p=3.021e-12


## 6. Convergent validity of the `approach` code

`approach` is a longitudinal *drive-up* behaviour. Testing it **within world-object clips only**
(A/B/D) avoids the HUD/verbal contamination. We also show the *contaminated* version to document why
the naive all-clips test overstates the road-gaze association.

In [7]:
def mw(frame, group, tele):
    g1=frame[frame[group]==1][tele].dropna(); g0=frame[frame[group]==0][tele].dropna()
    _,p=mannwhitneyu(g1,g0)
    return f"{group}=1 median={g1.median():.2f} (n={len(g1)}) vs =0 median={g0.median():.2f} (n={len(g0)})  p={p:.4g}"

world_clips = m[m.set.isin(['A','B','D'])]
print('WITHIN world-object clips (valid):')
print('  approach vs speed   :', mw(world_clips,'approach','During_Mean_Speed_mph'))
print('  approach vs diffuse off-object gaze:', mw(world_clips,'approach','During_Diffuse_Gaze_Pct'))
print('\nALL clips incl. Name/HUD (CONFOUNDED, for illustration only):')
print('  approach vs diffuse off-object gaze:', mw(m,'approach','During_Diffuse_Gaze_Pct'))

sub = m[(m.latency_s.notna()) & (m.precue_fixation_0_1_n==0) & (m.Target_Latency.notna())]
r,p = spearmanr(sub.latency_s, sub.Target_Latency)
print(f'\nvideo gaze-latency vs telemetry response-latency (first-acquisition, n={len(sub)}): r={r:.2f}, p={p:.4g}')

WITHIN world-object clips (valid):
  approach vs speed   : approach=1 median=1.50 (n=40) vs =0 median=1.90 (n=66)  p=0.05509
  approach vs road-gaze: approach=1 median=0.50 (n=40) vs =0 median=0.45 (n=66)  p=0.8069

ALL clips incl. Name/HUD (CONFOUNDED, for illustration only):
  approach vs road-gaze: approach=1 median=0.50 (n=40) vs =0 median=0.67 (n=102)  p=0.00436

video gaze-latency vs telemetry response-latency (first-acquisition, n=83): r=0.30, p=0.006077


## 7. Summary & caveats

* **A vs B** (clean, no HUD confound): contacts co-occur with more approach (69% vs 26%, p≈.017) and
  line-departure (92% vs 52%, p≈.025). Lean / dual-zoom n.s. at this N. → mechanism→outcome link.
* **D**: the behaviour recurs on Route 3 without collisions → not a Route-2 artefact.
* **Target-type specificity**: carried by telemetry road-gaze (world ≈0.47 vs Name/HUD ≈0.90, p<.0001),
  **not** by video approach (Set C is HUD-invalid for approach).
* **`approach` convergent validity**: within world-object clips it tracks lower speed only marginally
  and does **not** track road-gaze, the strong association vanishes once HUD/verbal clips are removed.
  So `approach` ≈ a drive-up/longitudinal behaviour tied to crashes, *not* an eyes-off-road proxy.
* **`apparent_lean_in`** is a soft posture proxy (head-mounted cam) and is non-significant throughout.
* **Single rater**: intra-rater test–retest (the `RECODE` set) is the outstanding reliability step.